# Inter-rater reliability of condition coding

Compare the primary condition codes with a second rater on 810 responses stratified by vignette and time point.

## Load the second-rater files

Returned codes use 1 for correct and 2 for incorrect. Primary codes use 1 and 0. A returned 2 is recoded to 0 before the comparison.

## Load packages and configure paths

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.inter_rater import cohens_kappa

ROOT = Path("..").resolve()
DATA_PATH = ROOT / "data" / "Psy_Case_Vignette_Data_Prolific_analysis.xlsx"
KEY_PATH = ROOT / "output" / "condition_second_rater_key.xlsx"
SECOND_PATH = ROOT / "output" / "interrater_reliability_condition.xlsx"
OUT_PATH = ROOT / "output" / "interrater_reliability.xlsx"

VIGNETTES = ["A", "B", "C", "D", "E"]
PHASES = ["pre", "post"]
PER_CELL = 81

CODE_LABEL = {0: "incorrect", 1: "correct"}


## Align the two ratings

Primary codes are read from the analysis file. The key only links `rating_id` to the participant, vignette, and phase. The link is checked against `manual_correct` in the key.

In [ ]:
def clean_note(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


key = pd.read_excel(KEY_PATH, sheet_name="Key")
returned = pd.read_excel(SECOND_PATH, sheet_name="Responses")
wide = pd.read_excel(DATA_PATH)

if wide["prolific_id"].duplicated().any():
    raise AssertionError("prolific_id is not unique in the analysis file")
if not key["rating_id"].is_unique or not returned["rating_id"].is_unique:
    raise AssertionError("rating_id is not unique")

returned = returned.copy()
if returned["correct"].isna().any():
    raise AssertionError(f"{int(returned['correct'].isna().sum())} responses have no second-rater code")
if set(returned["correct"].astype(int).unique()) != {1, 2}:
    raise AssertionError(f"Unexpected second-rater codes: {sorted(returned['correct'].unique())}")
returned["second_correct"] = returned["correct"].astype(int).map({1: 1, 2: 0})
returned["note"] = returned["note"].map(clean_note)

current_rows = []
for vignette in VIGNETTES:
    for phase in PHASES:
        column = f"vig_{vignette}_{phase}_condition_correct"
        part = wide[["prolific_id", column]].rename(columns={column: "primary_correct"})
        part["vignette"] = vignette
        part["phase"] = phase
        current_rows.append(part)
current = pd.concat(current_rows, ignore_index=True)

paired = key.merge(
    returned[["rating_id", "list", "response", "second_correct", "note"]],
    on="rating_id",
    how="outer",
    indicator=True,
    validate="one_to_one",
)
if not (paired["_merge"] == "both").all():
    raise AssertionError(paired["_merge"].value_counts().to_dict())
paired = paired.drop(columns="_merge")
if not (paired["list"] == paired["vignette"]).all():
    raise AssertionError("List letter does not match the vignette in the key")

paired = paired.merge(
    current,
    on=["prolific_id", "vignette", "phase"],
    how="left",
    validate="many_to_one",
)
if paired["primary_correct"].isna().any():
    raise AssertionError(f"{int(paired['primary_correct'].isna().sum())} sampled rows are missing from the analysis file")
if not paired["primary_correct"].isin([0, 1]).all():
    raise AssertionError("Primary codes are not limited to 0 and 1")
n_key_mismatch = int((paired["manual_correct"].astype(int) != paired["primary_correct"].astype(int)).sum())
if n_key_mismatch:
    raise AssertionError(f"Key manual_correct differs from the analysis file on {n_key_mismatch} rows")

cell_counts = paired.groupby(["vignette", "phase"]).size()
if len(cell_counts) != len(VIGNETTES) * len(PHASES) or not (cell_counts == PER_CELL).all():
    raise AssertionError(cell_counts.to_dict())

ratings = paired[
    [
        "rating_id",
        "vignette",
        "phase",
        "arm_label",
        "response",
        "primary_correct",
        "second_correct",
        "note",
    ]
].copy()
ratings["primary_correct"] = ratings["primary_correct"].astype(int)
ratings["second_correct"] = ratings["second_correct"].astype(int)
ratings["vignette"] = pd.Categorical(ratings["vignette"], categories=VIGNETTES, ordered=True)
ratings["phase"] = pd.Categorical(ratings["phase"], categories=PHASES, ordered=True)
ratings = ratings.sort_values(["vignette", "phase", "rating_id"]).reset_index(drop=True)

print(f"Paired responses: {len(ratings)}")
print(ratings.groupby(["vignette", "phase"], observed=True).size().unstack("phase"))


## Agreement, kappa, and marginal difference

Percent agreement is the share of responses with the same code. Cohen's kappa is unweighted. The 95% interval is the normal approximation from the asymptotic standard error, truncated to the range of kappa (−1 to 1). The exact McNemar test asks whether the two raters code a response correct equally often.

In [ ]:
def summarise(group: pd.DataFrame) -> dict:
    table = pd.crosstab(group["primary_correct"], group["second_correct"])
    table = table.reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    counts = table.to_numpy()
    n = int(counts.sum())
    n_agree = int(np.trace(counts))
    kappa = cohens_kappa(counts)
    return {
        "n": n,
        "n_agree": n_agree,
        "pct_agree": round(n_agree / n * 100, 2),
        "kappa": round(float(kappa.kappa), 4),
        "kappa_low": round(max(-1.0, float(kappa.kappa_low)), 4),
        "kappa_upp": round(min(1.0, float(kappa.kappa_upp)), 4),
        "pct_primary_correct": round(group["primary_correct"].mean() * 100, 2),
        "pct_second_correct": round(group["second_correct"].mean() * 100, 2),
        "n_primary_incorrect_second_correct": int(((group["primary_correct"] == 0) & (group["second_correct"] == 1)).sum()),
        "n_primary_correct_second_incorrect": int(((group["primary_correct"] == 1) & (group["second_correct"] == 0)).sum()),
    }


def summarise_by(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for key, group in data.groupby(columns, observed=True, sort=True):
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(columns, key))
        row.update(summarise(group))
        rows.append(row)
    return pd.DataFrame(rows)


overall = pd.DataFrame([summarise(ratings)])
crosstab = pd.crosstab(
    ratings["primary_correct"].map(CODE_LABEL).astype(pd.CategoricalDtype(["incorrect", "correct"], ordered=True)),
    ratings["second_correct"].map(CODE_LABEL).astype(pd.CategoricalDtype(["incorrect", "correct"], ordered=True)),
    dropna=False,
)
crosstab.index.name = "primary"
crosstab.columns.name = "second rater"

mcnemar_table = pd.crosstab(ratings["primary_correct"], ratings["second_correct"])
mcnemar_table = mcnemar_table.reindex(index=[0, 1], columns=[0, 1], fill_value=0)
mcnemar_result = mcnemar(mcnemar_table.to_numpy(), exact=True)
overall["mcnemar_statistic"] = int(mcnemar_result.statistic)
overall["mcnemar_p"] = float(mcnemar_result.pvalue)

row = overall.iloc[0]
print("Crosstab (rows = primary, columns = second rater)")
print(crosstab.to_string())
print()
print(f"Agreement: {int(row['n_agree'])} / {int(row['n'])} ({row['pct_agree']:.2f}%)")
print(
    f"Cohen's kappa: {row['kappa']:.4f} "
    f"(95% CI {row['kappa_low']:.4f} to {row['kappa_upp']:.4f})"
)
print(f"Primary coded correct: {row['pct_primary_correct']:.2f}%")
print(f"Second rater coded correct: {row['pct_second_correct']:.2f}%")
print(
    "Primary incorrect, second correct: "
    f"{int(row['n_primary_incorrect_second_correct'])}"
)
print(
    "Primary correct, second incorrect: "
    f"{int(row['n_primary_correct_second_incorrect'])}"
)
print(f"McNemar exact test: p = {row['mcnemar_p']:.4g}")


## Agreement by vignette and time

In [ ]:
by_vignette = summarise_by(ratings, ["vignette"])
by_phase = summarise_by(ratings, ["phase"])
by_vignette_phase = summarise_by(ratings, ["vignette", "phase"])

print("By vignette")
print(by_vignette.to_string(index=False))
print()
print("By time")
print(by_phase.to_string(index=False))
print()
print("By vignette and time")
print(by_vignette_phase.to_string(index=False))

lowest = by_vignette.loc[by_vignette["kappa"].idxmin()]
print()
print(
    f"Lowest kappa: vignette {lowest['vignette']} "
    f"({lowest['pct_agree']:.2f}% agreement, kappa {lowest['kappa']:.4f}). "
    f"Primary correct {lowest['pct_primary_correct']:.2f}%, "
    f"second rater correct {lowest['pct_second_correct']:.2f}%."
)


## Disagreements

Rows where the two codes differ, plus every response on which the second rater wrote a note. `rating_id` joins back to the key.

In [ ]:
DISAGREEMENT_COLS = [
    "rating_id",
    "vignette",
    "phase",
    "arm_label",
    "response",
    "primary_correct",
    "second_correct",
    "note",
]

disagreements = ratings.loc[
    ratings["primary_correct"] != ratings["second_correct"], DISAGREEMENT_COLS
].reset_index(drop=True)
notes = ratings.loc[ratings["note"].ne(""), DISAGREEMENT_COLS].reset_index(drop=True)

print(f"Disagreements: {len(disagreements)}")
print(
    disagreements.groupby(["vignette", "phase"], observed=True)
    .size()
    .rename("n")
    .reset_index()
    .to_string(index=False)
)
print()
print(f"Notes from the second rater: {len(notes)}")
if len(notes):
    print(notes.to_string(index=False))
print()
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", len(disagreements))
print(disagreements.to_string(index=False))


## Export

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_PATH) as writer:
    overall.to_excel(writer, sheet_name="Overall", index=False)
    crosstab.to_excel(writer, sheet_name="Crosstab")
    by_vignette.to_excel(writer, sheet_name="By vignette", index=False)
    by_phase.to_excel(writer, sheet_name="By phase", index=False)
    by_vignette_phase.to_excel(writer, sheet_name="By vignette and phase", index=False)
    disagreements.to_excel(writer, sheet_name="Disagreements", index=False)
    notes.to_excel(writer, sheet_name="Notes", index=False)

print(f"Saved {OUT_PATH}")
